# Large Language Model Basics

**Following the tutorial** : https://www.youtube.com/watch?v=UU1WVnMk4E8

## 1. Loading the Dataset

In [1]:
with open("dataset/wizard_of_oz.txt", "r", encoding="utf-8") as f:
    text = f.read()
print(text[:200])

    DOROTHY AND THE WIZARD IN OZ

  BY

  L. FRANK BAUM

  AUTHOR OF THE WIZARD OF OZ, THE LAND OF OZ, OZMA OF OZ, ETC.

  ILLUSTRATED BY JOHN R. NEILL

  BOOKS OF WONDER WILLIAM MORROW & CO., INC. NE


In [2]:
chars = sorted(set(text))
vocabulary_len = len(chars)
print(chars)

['\n', ' ', '!', '"', '&', "'", '(', ')', '*', ',', '-', '.', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', '[', ']', '_', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']


In [3]:
encode_chars = {c:i for i,c in enumerate(chars)}
decode_chars = {i:c for i,c in enumerate(chars)}

def encode(text):
    return [encode_chars.get(i) for i in text]

def decode(char_list):
    return ''.join([decode_chars.get(i) for i in char_list]) 
    

In [4]:
enc = encode("hello world")
print(f"Encoding: {enc} , \nDecoding: {decode(enc)}")

Encoding: [61, 58, 65, 65, 68, 1, 76, 68, 71, 65, 57] , 
Decoding: hello world


## 2. pytorch nn

In [5]:
import torch

In [6]:
data = torch.tensor(encode(text), dtype=torch.long)
data[:100]

tensor([ 1,  1,  1,  1, 28, 39, 42, 39, 44, 32, 49,  1, 25, 38, 28,  1, 44, 32,
        29,  1, 47, 33, 50, 25, 42, 28,  1, 33, 38,  1, 39, 50,  0,  0,  1,  1,
        26, 49,  0,  0,  1,  1, 36, 11,  1, 30, 42, 25, 38, 35,  1, 26, 25, 45,
        37,  0,  0,  1,  1, 25, 45, 44, 32, 39, 42,  1, 39, 30,  1, 44, 32, 29,
         1, 47, 33, 50, 25, 42, 28,  1, 39, 30,  1, 39, 50,  9,  1, 44, 32, 29,
         1, 36, 25, 38, 28,  1, 39, 30,  1, 39])

In [7]:
n = int(0.8*len(data))
train = data[:n]
test = data[n:]
print(f"Train len :{len(train)} \nTest len: {len(test)}")

n = int(0.8*len(train))
val = train[n:]
train = train[:n]
print(f"Train len :{len(train)} \nVal len: {len(val)}")

Train len :185984 
Test len: 46496
Train len :148787 
Val len: 37197


In [8]:
block_size = 8
x = train[:block_size]
y = train[1:block_size+1]

for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"When input is {context} target is {target}")

When input is tensor([1]) target is 1
When input is tensor([1, 1]) target is 1
When input is tensor([1, 1, 1]) target is 1
When input is tensor([1, 1, 1, 1]) target is 28
When input is tensor([ 1,  1,  1,  1, 28]) target is 39
When input is tensor([ 1,  1,  1,  1, 28, 39]) target is 42
When input is tensor([ 1,  1,  1,  1, 28, 39, 42]) target is 39
When input is tensor([ 1,  1,  1,  1, 28, 39, 42, 39]) target is 44


In [9]:
batch_size = 4
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"device : {device}")

def get_batch(split):
    d = train if split == "train" else val
    ix = torch.randint(len(d) - block_size, (batch_size,))
    #print(ix)
    x = torch.stack([d[i:(i+block_size)] for i in ix])
    y = torch.stack([d[i+1:(i+block_size) + 1] for i in ix])

    x,y = x.to(device=device), y.to(device=device)
    return x,y

x,y = get_batch("train")
print(f"input : {x} \ntarget : {y}")

device : cpu
input : tensor([[ 9,  1, 76, 58,  1, 69, 71, 58],
        [67, 68, 76,  9,  1, 54, 67, 57],
        [65, 68, 67, 60,  9,  3,  1, 71],
        [ 1, 67, 58, 75, 58, 71,  0, 55]]) 
target : tensor([[ 1, 76, 58,  1, 69, 71, 58, 59],
        [68, 76,  9,  1, 54, 67, 57,  0],
        [68, 67, 60,  9,  3,  1, 71, 58],
        [67, 58, 75, 58, 71,  0, 55, 58]])


In [10]:
import torch.nn as nn
import torch.nn.functional as F

#### Documentation
 
a. nn.Embedding => https://pytorch.org/docs/stable/generated/torch.nn.Embedding.html

b. nn.CrossEntropyLoss => https://pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html#torch.nn.CrossEntropyLoss

c. nn.Layer => https://pytorch.org/docs/stable/generated/torch.nn.Linear.html#torch.nn.Linear

In [11]:
# exploring embedding , it has reandomly init weights
emb = nn.Embedding(num_embeddings=10, embedding_dim=3)

context = torch.tensor([1])
print(f"Context {context}, embedding  for the context: {emb(context)}")

context = torch.tensor([2])
print(f"Context {context}, embedding  for the context: {emb(context)}")

try:
    context = torch.tensor([10])
    print(f"Context {context}, embedding  for the context: {emb(context)}")
except:
    print("Fail")

Context tensor([1]), embedding  for the context: tensor([[ 1.2423, -0.2055, -1.2133]], grad_fn=<EmbeddingBackward0>)
Context tensor([2]), embedding  for the context: tensor([[-1.2460,  1.2592,  0.5635]], grad_fn=<EmbeddingBackward0>)
Fail


In [12]:
# if the index tensor is has inc dim, results reflects it as well
context = torch.tensor([1])
ex = emb(context)
print(f"Context {context}, embedding {ex}, shape {ex.shape}")

context = torch.tensor([[1]])
ex = emb(context)
print(f"Context {context}, embedding {ex}, shape {ex.shape}")

Context tensor([1]), embedding tensor([[ 1.2423, -0.2055, -1.2133]], grad_fn=<EmbeddingBackward0>), shape torch.Size([1, 3])
Context tensor([[1]]), embedding tensor([[[ 1.2423, -0.2055, -1.2133]]], grad_fn=<EmbeddingBackward0>), shape torch.Size([1, 1, 3])


In [13]:
# if using pretrained weights
weight = torch.FloatTensor([[1, 2.3, 3], [4, 5.1, 6.3]])
embedding = nn.Embedding.from_pretrained(weight)
input = torch.LongTensor([1])
embedding(input)

tensor([[4.0000, 5.1000, 6.3000]])

In [14]:
# view
a = torch.randn(4, 2)
print(f"a : {a},\nwith view {a.view(8)}")
print(f"a : {a},\nwith view {a.view((2,4))}")

a : tensor([[ 0.1621,  1.3840],
        [-0.2630, -0.7580],
        [-0.3107, -0.1474],
        [ 0.7146,  0.6655]]),
with view tensor([ 0.1621,  1.3840, -0.2630, -0.7580, -0.3107, -0.1474,  0.7146,  0.6655])
a : tensor([[ 0.1621,  1.3840],
        [-0.2630, -0.7580],
        [-0.3107, -0.1474],
        [ 0.7146,  0.6655]]),
with view tensor([[ 0.1621,  1.3840, -0.2630, -0.7580],
        [-0.3107, -0.1474,  0.7146,  0.6655]])


In [15]:
# eval_interval = 2500
learning_rate = 3e-4
eval_iters = 250

In [16]:
class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)
        
    def forward(self, index, targets=None):
        logits = self.token_embedding_table(index)
        
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C) # same as faltening it, eg: emb of shape (2, 4, 60), will be changed to (8, 60)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)  # cross entroy requires batch size first then the number of classes
        
        return logits, loss
    
    def generate(self, index, max_new_tokens):
        # index is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # get the predictions
            logits, loss = self.forward(index)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            index_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            index = torch.cat((index, index_next), dim=1) # (B, T+1)
        return index

model = BigramLanguageModel(vocab_size=vocabulary_len)
m = model.to(device)

context = torch.zeros((1,1), dtype=torch.long, device=device)
generated_chars = decode(m.generate(context, max_new_tokens=500)[0].tolist())
print(generated_chars)


X-N!YQDHCS1thMkmy3Lf5PxBASD*F6SNSN&qMPdY6)_pdXo!vWhta,-J &P9BUo_(R8Nq]&VS6'mEPHTHUVgIy.YmMMN"R
;USjBhzUjSrd,2_w1zfvI09 hV8B8f2.fbh?'A3(6Ncdx5*N3H3_8HCi6!),PvPX1GlmEMpE(kv*8OZz;'COLl2"IIswmEb,!Rb7Z-Gk
fny
(N"6EVy,yk]k1b_B5_tO1&PstqTHbC9B_hELjoW4uD]DIihM41C-hV!X*OPYT:Atm"K&*
v"QU' !It97.uq(]D*xqXb'uaaut?auGfW.*x_rc Xb6?Zi8A1&Olmi?NN&j_?o2 Xl5r7ZoumJmiUk;?wXm9IPZ3_a]u1G:?vh53t.pNHCj&w3)*s)*LI:
k*]jq24GF)i2]DtF&BW6H,CG":*XsPf5B8KsHlPC,"uh?a'E.(&*sITA;]HUc .Ovh6SYwSvhKxqC[f?iBgT]&v7uc6Pf5VuRIWBz:D1Sf


In [17]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

In [18]:
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

max_iters = 50000

for iter in range(max_iters):
    if iter % 500 == 0:
        losses = estimate_loss()
        print(f"step: {iter}, train loss: {losses['train']:.3f}, val loss: {losses['val']:.3f}")

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model.forward(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
print(loss.item())

step: 0, train loss: 4.873, val loss: 4.862
step: 500, train loss: 4.732, val loss: 4.749
step: 1000, train loss: 4.632, val loss: 4.635
step: 1500, train loss: 4.507, val loss: 4.508
step: 2000, train loss: 4.380, val loss: 4.390
step: 2500, train loss: 4.270, val loss: 4.280
step: 3000, train loss: 4.176, val loss: 4.173
step: 3500, train loss: 4.080, val loss: 4.093
step: 4000, train loss: 3.987, val loss: 3.982
step: 4500, train loss: 3.896, val loss: 3.929
step: 5000, train loss: 3.813, val loss: 3.819
step: 5500, train loss: 3.734, val loss: 3.747
step: 6000, train loss: 3.668, val loss: 3.686
step: 6500, train loss: 3.583, val loss: 3.586
step: 7000, train loss: 3.506, val loss: 3.536
step: 7500, train loss: 3.443, val loss: 3.467
step: 8000, train loss: 3.387, val loss: 3.427
step: 8500, train loss: 3.325, val loss: 3.330
step: 9000, train loss: 3.281, val loss: 3.288
step: 9500, train loss: 3.197, val loss: 3.232
step: 10000, train loss: 3.167, val loss: 3.184
step: 10500, tra

In [19]:
context = torch.zeros((1,1), dtype=torch.long, device=device)
generated_chars = decode(m.generate(context, max_new_tokens=100)[0].tolist())
print(generated_chars)


"
fencord waingle th wang we tinots.
bowar ang oicBNg theancor m beatlotskife ppr aro ong ohey, stou


In [20]:
enc = encode("hor")
decode_ls = decode(m.generate(torch.tensor([enc]), max_new_tokens=5)[0].tolist())
print(f"Next 5 tokens : {decode_ls}")

enc = encode("The boy")
decode_ls = decode(m.generate(torch.tensor([enc]), max_new_tokens=10)[0].tolist())
print(f"Next 10 tokens : {decode_ls}")

Next 5 tokens : horese w
Next 10 tokens : The boy he mulend
